# Model Training — Random Forest (8 Attack Categories)

Notebook ini melatih model **Random Forest** menggunakan dataset yang sudah di-mapping ke **8 kategori serangan siber** yang sesuai dengan *knowledge base* sistem pakar (CF Rules):

| # | Attack Type |
|---|---|
| 1 | Reconnaissance |
| 2 | Brute Force Attack |
| 3 | SQL Injection |
| 4 | Distributed Denial of Service (DDoS) |
| 5 | Cross-Site Scripting (XSS) |
| 6 | Man-in-the-Middle (MitM) |
| 7 | Phishing |
| 8 | Ransomware |

Silakan klik **Run All**.

### 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

### 2. Setup Direktori (Auto-Detect: Lokal, Kaggle, Colab)

In [ ]:
if os.path.exists('/kaggle'):
    print("Environment: Kaggle Detected")
    csv_paths = list(Path("/kaggle/input").rglob("5_mapped_attacks.csv"))
    DATASET_DIR = csv_paths[0].parent if csv_paths else Path("/kaggle/input")
    MODEL_DIR = Path("/kaggle/working/models")

elif os.path.exists('/content'):
    print("Environment: Google Colab Detected")
    DATASET_DIR = Path.cwd()
    MODEL_DIR = Path.cwd() / "models"

else:
    print("Environment: Local Machine Detected")
    BASE_DIR = Path.cwd()
    DATASET_DIR = BASE_DIR / "dataset"
    MODEL_DIR = BASE_DIR / "app" / "ml" / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH        = MODEL_DIR / "classifier.pkl"
ENCODER_PATH      = MODEL_DIR / "label_encoder.pkl"
FEATURE_LIST_PATH = MODEL_DIR / "feature_list.pkl"

print(f"Dataset Directory : {DATASET_DIR}")
print(f"Model Directory   : {MODEL_DIR}")

### 3. Load Dataset
Dataset `5_mapped_attacks.csv` berisi ~2956 baris dengan 8 kelas serangan yang seimbang. Setiap baris berisi kombinasi gejala (*symptoms*) yang dipetakan ke jenis serangan tertentu.

In [ ]:
df = pd.read_csv(DATASET_DIR / "5_mapped_attacks.csv")

print(f"Total data: {len(df)} rows")
print(f"\nDistribusi label:")
print(df['label'].value_counts())

### 4. Visualisasi Distribusi Kelas

In [ ]:
label_counts = df['label'].value_counts()

plt.figure(figsize=(12, 6))
bars = sns.barplot(x=label_counts.values, y=label_counts.index, palette="viridis")
plt.title("Distribusi Sampel per Kategori Serangan", fontsize=14)
plt.xlabel("Jumlah Sampel")
for i, v in enumerate(label_counts.values):
    plt.text(v + 2, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 5. Ekstraksi Fitur (MultiLabelBinarizer)
Setiap gejala dalam kolom `features` dipisahkan oleh koma, lalu diubah menjadi matriks *one-hot encoding* menggunakan `MultiLabelBinarizer`.

In [ ]:
print("Extracting and encoding features (symptoms)...")
tags_list = [[t.strip() for t in str(s).split(",")] for s in df["features"]]

mlb = MultiLabelBinarizer()
X = mlb.fit_transform(tags_list)
all_symptoms = list(mlb.classes_)

print(f"Total unique symptoms (features): {len(all_symptoms)}")
print(f"Sample symptoms: {all_symptoms[:10]}")

### 6. Encoding Label & Split Data (Stratified 80/20)
Karena 8 kelas sudah seimbang, kita bisa menggunakan `stratify=y` dengan aman untuk memastikan distribusi label yang proporsional di training dan test set.

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df["label"].str.strip())

print(f"Label classes: {list(le.classes_)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining set : {X_train.shape[0]} data")
print(f"Testing set  : {X_test.shape[0]} data")

### 7. Training Random Forest (Dioptimalkan)
Parameter yang digunakan:
- `n_estimators=500` — 500 pohon keputusan untuk prediksi lebih stabil
- `class_weight='balanced'` — imbangkan pengaruh tiap kelas
- `max_features='sqrt'` — kurangi korelasi antar pohon
- `random_state=42` — hasil reproducible

In [ ]:
print("Training RandomForestClassifier...")
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print("Training completed.")

### 8. Evaluasi: Test Accuracy & 5-Fold Cross-Validation
**Test Accuracy** menunjukkan performa pada data yang belum pernah dilihat model. **CV Accuracy** adalah rata-rata dari 5 percobaan split berbeda — ini adalah *validation accuracy* yang paling andal.

In [ ]:
# Test Accuracy
y_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy         : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# 5-Fold Stratified Cross-Validation
print("\nRunning 5-Fold Stratified Cross-Validation...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=skf, scoring="accuracy", n_jobs=-1)

print(f"CV Fold Scores        : {[f'{s:.4f}' for s in cv_scores]}")
print(f"CV Mean (Validation)  : {cv_scores.mean():.4f} ({cv_scores.mean()*100:.2f}%)")
print(f"CV Std                : +/- {cv_scores.std():.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

### 9. Visualisasi: CV Score + Confusion Matrix + Feature Importances

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Plot 1: CV Score per Fold
fold_labels = [f"Fold {i+1}" for i in range(len(cv_scores))]
colors = ['green' if s >= 0.96 else 'orange' if s >= 0.90 else 'red' for s in cv_scores]
axes[0].bar(fold_labels, cv_scores, color=colors, edgecolor='black')
axes[0].axhline(cv_scores.mean(), color='blue', linestyle='--', label=f'Mean: {cv_scores.mean():.2%}')
axes[0].set_ylim(0, 1.05)
axes[0].set_title("5-Fold Cross-Validation Accuracy", fontsize=12)
axes[0].set_ylabel("Accuracy")
axes[0].legend()
for i, v in enumerate(cv_scores):
    axes[0].text(i, v + 0.005, f'{v:.2%}', ha='center', fontsize=9, fontweight='bold')

# Plot 2: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[c[:12] for c in le.classes_],
            yticklabels=[c[:12] for c in le.classes_],
            ax=axes[1])
axes[1].set_title("Confusion Matrix", fontsize=12)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
axes[1].tick_params(axis='x', rotation=45)

# Plot 3: Top 15 Feature Importances
importances = model.feature_importances_
indices = np.argsort(importances)[::-1][:15]
top_features = [all_symptoms[i] for i in indices]
sns.barplot(x=importances[indices], y=top_features, palette="magma", ax=axes[2])
axes[2].set_title("Top 15 Most Important Symptoms", fontsize=12)
axes[2].set_xlabel("Relative Importance")

plt.suptitle("Random Forest Model Evaluation", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 10. Simpan Model ke Backend FastAPI
Ketiga file model disimpan ke lokasi yang sesuai dengan environment yang terdeteksi.

In [ ]:
print(f"Saving models to {MODEL_DIR}...")
joblib.dump(model, MODEL_PATH)
joblib.dump(le, ENCODER_PATH)
joblib.dump(all_symptoms, FEATURE_LIST_PATH)
print("[OK] All models saved successfully.")
print(f"  - {MODEL_PATH}")
print(f"  - {ENCODER_PATH}")
print(f"  - {FEATURE_LIST_PATH}")

### 11. Tes Prediksi Manual
Uji model dengan gejala yang sama persis seperti yang digunakan di CF rules.

In [ ]:
def test_prediction(symptoms):
    print(f"Input   : {symptoms}")
    feature_vector = {s: 0 for s in all_symptoms}
    matched = [s for s in symptoms if s in feature_vector]
    for s in matched:
        feature_vector[s] = 1

    X_input = pd.DataFrame([feature_vector])
    predicted_class = model.predict(X_input)[0]
    probabilities   = model.predict_proba(X_input)[0]
    confidence      = float(max(probabilities))
    attack_type     = le.inverse_transform([predicted_class])[0]

    print(f"Matched : {matched}")
    print(f"Attack  : {attack_type}")
    print(f"Conf.   : {confidence:.2%}")
    print()

print("--- Uji Prediksi Manual ---")
test_prediction(["port_scanning", "os_fingerprinting"])           # -> Reconnaissance
test_prediction(["sql_injection_pattern", "unusual_db_query"])     # -> SQL Injection
test_prediction(["brute_force_login", "multiple_failed_auth"])     # -> Brute Force
test_prediction(["mass_file_encryption", "ransom_note_created"])   # -> Ransomware
test_prediction(["xss_payload_detected", "cookie_theft_attempt"])  # -> XSS
test_prediction(["arp_spoofing", "ssl_stripping"])                 # -> MitM